# 02 — Prompt Engineering

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 02_Modelos_PreTreinados

---

## O que você vai aprender

- **Zero-shot** — perguntar direto, sem exemplos
- **Few-shot** — dar exemplos antes de pedir
- **Chain-of-Thought (CoT)** — fazer o modelo raciocinar passo a passo
- **Role prompting** — definir um papel específico para o modelo
- **Output formatting** — controlar o formato da saída (JSON, markdown, tabelas)
- **Prompt templates** — estrutura reutilizável para o projeto

---

> 💡 Prompt engineering é a habilidade de **comunicar intenções claramente** para um LLM.  
> Um bom prompt pode dobrar a qualidade da resposta sem custar nenhum token extra de modelo.

## Setup

In [9]:
import sys, os, json
sys.path.append(os.path.abspath('..'))

from shared.llm_factory import chat, chat_stream, get_provider_info

info = get_provider_info()
print(f"Provider : {info['provider']} | Modelo: {info['model']}")

Provider : deepseek | Modelo: deepseek-chat


---
## 1. Zero-shot — sem exemplos

A forma mais simples: você faz a pergunta diretamente, sem demonstrar o que quer.  
Funciona bem para tarefas simples — mas pode ser impreciso em tarefas mais específicas.

In [2]:
# Zero-shot simples
resposta = chat("Classifique o sentimento: 'O modelo demorou mas o resultado foi excelente'")
print("Zero-shot simples:")
print(resposta)

Zero-shot simples:
**Sentimento: Positivo**  

**Justificativa:**  
Apesar de mencionar uma crítica leve ("demorou"), o foco da frase está na avaliação final positiva ("resultado excelente"). A conjunção "mas" indica que o aspecto negativo é superado pelo desfecho favorável, resultando em um tom geral positivo.


In [3]:
# Zero-shot com instrução mais clara — já melhora bastante
resposta = chat(
    "Classifique o sentimento do texto abaixo.\n"
    "Responda APENAS com uma palavra: Positivo, Negativo ou Neutro.\n\n"
    "Texto: 'O modelo demorou mas o resultado foi excelente'"
)
print("Zero-shot com instrução clara:")
print(resposta)

Zero-shot com instrução clara:
Positivo


---
## 2. Few-shot — ensinando com exemplos

Você mostra **exemplos do padrão esperado** antes de fazer a pergunta real.  
É especialmente poderoso quando o formato de saída precisa ser exato.

In [4]:
few_shot_prompt = """
Classifique o sentimento de textos de avaliação de modelos de IA.
Responda no formato: SENTIMENTO | CONFIANÇA (alta/média/baixa) | MOTIVO

Exemplos:
Texto: "A API respondeu em 200ms, impressionante!"
Resposta: POSITIVO | alta | velocidade de resposta destacada

Texto: "Funcionou, mas os resultados variam muito."
Resposta: NEUTRO | média | funcionalidade confirmada mas inconsistência apontada

Texto: "Completamente inútil para o meu caso de uso."
Resposta: NEGATIVO | alta | inadequação explícita ao caso de uso

Agora classifique:
Texto: "O modelo entendeu meu contexto perfeitamente, mas inventou alguns fatos."
Resposta:"""

resposta = chat(few_shot_prompt)
print("Few-shot:")
print(resposta)

Few-shot:
NEGATIVO | alta | embora o entendimento do contexto seja elogiado, a invenção de fatos é um erro crítico e negativo


In [5]:
# Few-shot com múltiplos textos de uma vez
textos = [
    "Rodou na primeira tentativa, sem erros.",
    "Prometeu muito mas entregou pouco.",
    "Não é perfeito, mas resolve 80% dos casos.",
]

exemplos = """
Classifique o sentimento. Formato: SENTIMENTO | MOTIVO BREVE

Exemplos:
"Perfeito para produção" → POSITIVO | adequação confirmada
"Muito lento para uso real" → NEGATIVO | problema de performance
"Funciona às vezes" → NEUTRO | inconsistência
"""

for texto in textos:
    prompt = exemplos + f'\nTexto: "{texto}"\nResposta:'
    r = chat(prompt, temperature=0.0)
    print(f'  "{texto}"')
    print(f'  → {r.strip()}\n')

  "Rodou na primeira tentativa, sem erros."
  → POSITIVO | funcionamento imediato e sem problemas

  "Prometeu muito mas entregou pouco."
  → NEGATIVO | expectativa frustrada

  "Não é perfeito, mas resolve 80% dos casos."
  → NEUTRO | reconhecimento de limitação com utilidade



---
## 3. Chain-of-Thought (CoT) — raciocínio passo a passo

Pedir ao modelo para **pensar antes de responder** melhora muito a qualidade em tarefas que exigem raciocínio.  
A frase mágica: _"Pense passo a passo"_ ou _"Vamos resolver isso por etapas"_.

In [6]:
problema = """
Tenho um projeto de ML com as seguintes características:
- Dataset: 10.000 exemplos de texto em português
- Tarefa: classificação em 5 categorias
- Recurso disponível: apenas CPU, 8GB RAM
- Prazo: 2 semanas

Qual abordagem devo usar: fine-tuning de um modelo grande, 
RAG, ou um classificador clássico com TF-IDF?
"""

# Sem CoT
print("=== Sem Chain-of-Thought ===")
r_simples = chat(problema)
print(r_simples)

=== Sem Chain-of-Thought ===
Dadas as suas restrições, **um classificador clássico com TF-IDF + SVM/Logistic Regression** é a melhor escolha.  

Aqui está a análise detalhada:

---

### 1. **Fine-tuning de um modelo grande (BERT, RoBERTa, etc.)**
- **Prós**: Alta precisão se os dados forem complexos e o modelo pré-treinado for adequado ao português.
- **Contras**:
  - **Recursos**: Modelos grandes exigem GPU para treino eficiente; em CPU será muito lento (pode levar dias).
  - **Memória**: 8GB RAM pode não ser suficiente para carregar o modelo e os dados de treino.
  - **Prazo**: 2 semanas é curto para ajustes, testes e iterações com treino lento em CPU.
  - **Dados**: 10k exemplos podem ser suficientes, mas o custo computacional é alto.

---

### 2. **RAG (Retrieval-Augmented Generation)**
- **Prós**: Bom para tarefas que precisam de conhecimento externo, como QA.
- **Contras**:
  - **Complexidade**: Overkill para classificação simples em 5 categorias.
  - **Recursos**: Exige um model

In [11]:
# Com CoT — mesmo problema, melhor raciocínio
print("=== Com Chain-of-Thought ===")
r_cot = chat(
    problema,
    system="""Você é um especialista em ML. 
Ao responder, siga SEMPRE esta estrutura:
1. ANÁLISE DAS RESTRIÇÕES: liste os fatores limitantes
2. AVALIAÇÃO DAS OPÇÕES: analise cada opção contra as restrições  
3. RECOMENDAÇÃO: escolha clara com justificativa
4. PRÓXIMOS PASSOS: 3 ações concretas para começar"""
)
print(r_cot)

=== Com Chain-of-Thought ===
1. **ANÁLISE DAS RESTRIÇÕES**
   - **Recursos limitados**: Apenas CPU com 8GB RAM exclui modelos grandes (BERT, RoBERTa) no treinamento
   - **Dataset moderado**: 10.000 exemplos é suficiente para métodos clássicos, mas pequeno para fine-tuning profundo
   - **Prazo curto**: 2 semanas exige solução rápida de implementação e treinamento
   - **Tarefa específica**: Classificação multiclasse (5 categorias) em português

2. **AVALIAÇÃO DAS OPÇÕES**
   - **Fine-tuning de modelo grande**:
     - ❌ Impraticável: modelos transformer exigem GPU para treinamento em tempo viável
     - ❌ Memória insuficiente: 8GB RAM não suporta modelos grandes
     - ❌ Prazo incompatível: treinamento levaria dias/semanas em CPU
   
   - **RAG (Retrieval-Augmented Generation)**:
     - ❌ Overkill para classificação simples
     - ❌ Complexidade desnecessária para 5 categorias
     - ❌ Exigiria embeddings e LLM, ambos pesados para CPU
   
   - **Classificador clássico com TF-IDF**:
   

---
## 4. Role Prompting — dando um papel ao modelo

Definir **quem o modelo deve ser** muda completamente o tom, profundidade e foco da resposta.

In [12]:
pergunta = "Como devo estruturar meu projeto de IA para produção?"

papeis = {
    "Engenheiro Sênior": "Você é um engenheiro de software sênior com 10 anos de experiência em sistemas de IA em produção. Seja direto, técnico e prático.",
    "Professor Didático": "Você é um professor universitário de IA. Explique conceitos com analogias simples, adequado para iniciantes.",
    "Revisor Crítico"   : "Você é um revisor técnico exigente. Aponte os principais riscos e erros comuns que desenvolvedores cometem.",
}

for papel, system in papeis.items():
    print(f"\n{'='*55}")
    print(f"PAPEL: {papel}")
    print('='*55)
    r = chat(pergunta, system=system, max_tokens=200)
    print(r)


PAPEL: Engenheiro Sênior
**Estrutura essencial para IA em produção:**

## 1. **Arquitetura Base**
```
project/
├── src/
│   ├── data/          # Pipelines de dados
│   ├── models/        # Definições e treino
│   ├── inference/     # Serviço de predição
│   ├── monitoring/    # Métricas e logs
│   └── api/           # Endpoints REST/gRPC
├── tests/
├── deployments/       # Docker, Kubernetes
├── monitoring/        # Dashboards, alertas
└── docs/
```

## 2. **Componentes Críticos**

**Versionamento:**
- Dados: DVC ou similar
- Modelos: MLflow, Weights & Biases
- Código: Git com tags semânticas

**Serving:**
- Escolha baseada em latência: Triton

PAPEL: Professor Didático
Imagine que você está construindo uma casa, não apenas uma cabana temporária. Seu projeto de IA para produção é como construir uma casa sólida, com alicerce, encanamento e segurança. Vamos estruturar em camadas:

**1. Alicerce (Infraestrutura)**
- Escolha ferramentas confiáveis (TensorFlow/PyTorch, scikit-learn)
- Cont

---
## 5. Output Formatting — controlando o formato da saída

Uma das técnicas mais úteis na prática: pedir ao modelo que responda em um **formato específico e processável**.

In [13]:
# Saída em JSON estruturado
prompt_json = """
Analise o seguinte trecho de código Python e retorne um JSON com esta estrutura exata:
{
  "qualidade": "boa" | "media" | "ruim",
  "problemas": [lista de problemas encontrados],
  "sugestoes": [lista de melhorias],
  "nota": número de 0 a 10
}
Retorne APENAS o JSON, sem explicações.

Código:
def predict(data):
    model = load_model('model.pkl')
    result = model.predict(data)
    return result
"""

r = chat(prompt_json, temperature=0.0)
print("Resposta bruta:")
print(r)

# Tenta parsear como JSON
try:
    # Remove blocos markdown se o modelo incluir
    r_limpo = r.strip().replace("```json", "").replace("```", "").strip()
    dados = json.loads(r_limpo)
    print("\nJSON parseado com sucesso:")
    print(f"  Qualidade : {dados['qualidade']}")
    print(f"  Nota      : {dados['nota']}/10")
    print(f"  Problemas : {len(dados['problemas'])} encontrados")
    for p in dados['problemas']:
        print(f"    - {p}")
except json.JSONDecodeError as e:
    print(f"\nErro ao parsear JSON: {e}")

Resposta bruta:
{
  "qualidade": "media",
  "problemas": ["Ausência de tratamento de erros", "Caminho do modelo é fixo e pode não existir", "Nenhuma validação dos dados de entrada"],
  "sugestoes": ["Adicionar try-except para capturar exceções", "Validar se o arquivo 'model.pkl' existe antes de carregar", "Verificar formato e tipo dos dados de entrada", "Considerar usar caminhos de arquivo configuráveis"],
  "nota": 5
}

JSON parseado com sucesso:
  Qualidade : media
  Nota      : 5/10
  Problemas : 3 encontrados
    - Ausência de tratamento de erros
    - Caminho do modelo é fixo e pode não existir
    - Nenhuma validação dos dados de entrada


In [14]:
# Saída em tabela markdown
prompt_tabela = """
Compare os seguintes modelos de embedding para uso em português brasileiro.
Retorne uma tabela markdown com colunas: Modelo | Tamanho | Qualidade PT-BR | Custo | Indicado para

Modelos: 
- sentence-transformers/all-MiniLM-L6-v2
- sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  
- text-embedding-3-small (OpenAI)
- text-embedding-ada-002 (OpenAI)
"""

r = chat(prompt_tabela, max_tokens=400)
print(r)

Aqui está a comparação dos modelos de embedding para português brasileiro:

| Modelo | Tamanho | Qualidade PT-BR | Custo | Indicado para |
|--------|---------|----------------|-------|---------------|
| **sentence-transformers/all-MiniLM-L6-v2** | 80MB | ⭐⭐☆☆☆ (Básica) | Gratuito (self-hosted) | Projetos simples, recursos limitados, prototipagem rápida |
| **sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2** | 420MB | ⭐⭐⭐☆☆ (Boa) | Gratuito (self-hosted) | Aplicações multilíngues, qualidade melhor que MiniLM-L6, balanceamento custo-benefício |
| **text-embedding-3-small (OpenAI)** | API | ⭐⭐⭐⭐☆ (Muito Boa) | $0.02/milhão tokens | Produção, alta precisão, escalabilidade, sem infraestrutura própria |
| **text-embedding-ada-002 (OpenAI)** | API | ⭐⭐⭐☆☆ (Boa) | $0.10/milhão tokens | Sistemas legados, compatibilidade com aplicações existentes da OpenAI |

---

### **Análise Detalhada:**

#### **Modelos Open Source (Sentence Transformers):**
- **all-MiniLM-L6-v2**: Modelo leve em 

---
## 6. Prompt Templates — reutilizando estruturas

Em projetos reais, os prompts não são strings fixas — são **templates com variáveis**.  
Vamos criar um sistema de templates simples que usaremos no Assistente Técnico.

In [15]:
class PromptTemplate:
    """
    Template simples para prompts com variáveis.
    Usado em todo o projeto EAI_07.
    """

    def __init__(self, template: str, variaveis: list[str]):
        self.template = template
        self.variaveis = variaveis

    def formatar(self, **kwargs) -> str:
        # Verifica se todas as variáveis foram passadas
        faltando = [v for v in self.variaveis if v not in kwargs]
        if faltando:
            raise ValueError(f"Variáveis faltando no template: {faltando}")
        return self.template.format(**kwargs)

    def __repr__(self):
        return f"PromptTemplate(variáveis={self.variaveis})"


# ── Templates do projeto ─────────────────────────────────────

TEMPLATE_RAG = PromptTemplate(
    template="""
Use os trechos abaixo do projeto ESPECIALISTA_EM_IA para responder a pergunta.

CONTEXTO RECUPERADO:
{contexto}

PERGUNTA: {pergunta}

Se o contexto não for suficiente, diga claramente e responda com conhecimento geral.
""",
    variaveis=["contexto", "pergunta"]
)

TEMPLATE_REVISAO_CODIGO = PromptTemplate(
    template="""
Revise o código {linguagem} abaixo considerando:
- Boas práticas da linguagem
- Performance
- Legibilidade
- Possíveis bugs

Contexto do projeto: {contexto_projeto}

CÓDIGO:
```{linguagem}
{codigo}
```

Retorne: problemas encontrados e sugestões de melhoria.
""",
    variaveis=["linguagem", "codigo", "contexto_projeto"]
)

TEMPLATE_EXPLICACAO = PromptTemplate(
    template="""
Explique o conceito de "{conceito}" para um {nivel} em IA.
Use {estilo}.
Limite a resposta a {max_linhas} linhas.
""",
    variaveis=["conceito", "nivel", "estilo", "max_linhas"]
)

print("Templates criados:")
print(f"  TEMPLATE_RAG            : {TEMPLATE_RAG}")
print(f"  TEMPLATE_REVISAO_CODIGO : {TEMPLATE_REVISAO_CODIGO}")
print(f"  TEMPLATE_EXPLICACAO     : {TEMPLATE_EXPLICACAO}")

Templates criados:
  TEMPLATE_RAG            : PromptTemplate(variáveis=['contexto', 'pergunta'])
  TEMPLATE_REVISAO_CODIGO : PromptTemplate(variáveis=['linguagem', 'codigo', 'contexto_projeto'])
  TEMPLATE_EXPLICACAO     : PromptTemplate(variáveis=['conceito', 'nivel', 'estilo', 'max_linhas'])


In [16]:
# Usando o template de explicação
prompt = TEMPLATE_EXPLICACAO.formatar(
    conceito="embeddings",
    nivel="estudante com base em machine learning",
    estilo="uma analogia do mundo real seguida da explicação técnica",
    max_linhas=10
)
print("Prompt gerado:")
print(prompt)
print("\nResposta:")
print(chat(prompt))

Prompt gerado:

Explique o conceito de "embeddings" para um estudante com base em machine learning em IA.
Use uma analogia do mundo real seguida da explicação técnica.
Limite a resposta a 10 linhas.


Resposta:
Imagine que cada palavra é uma pessoa em uma festa.  
Pessoas com interesses parecidos (como "rei" e "rainha") ficam próximas, enquanto outras distantes (como "rei" e "computador").  

Tecnicamente, embeddings são representações numéricas densas de dados (como palavras ou imagens) em um espaço vetorial.  
O modelo de machine learning aprende a posicionar itens semanticamente similares próximos nesse espaço, capturando relações contextuais.  
Isso permite que algoritmos entendam significado e similaridade, sendo a base para tarefas como busca, tradução e recomendação.


In [17]:
# Usando o template de revisão de código — contexto real do projeto
codigo_exemplo = """
def indexar_projeto(caminho):
    arquivos = []
    for f in os.listdir(caminho):
        with open(f) as file:
            arquivos.append(file.read())
    return arquivos
"""

prompt = TEMPLATE_REVISAO_CODIGO.formatar(
    linguagem="python",
    codigo=codigo_exemplo,
    contexto_projeto="Indexador de arquivos do projeto ESPECIALISTA_EM_IA para RAG"
)

print(chat(prompt, max_tokens=400))

## Problemas encontrados no código:

1. **Bug crítico**: `os.listdir()` retorna apenas nomes de arquivos, não caminhos completos
   - O `open(f)` falhará se o arquivo não estiver no diretório atual de execução
   - Não considera subdiretórios

2. **Tratamento de erros inexistente**:
   - Não verifica se é arquivo ou diretório
   - Não trata erros de leitura (permissões, encoding, arquivos binários)
   - Não fecha arquivos corretamente em caso de exceção

3. **Performance**:
   - Lê todos os arquivos de uma vez na memória
   - Não filtra por extensões relevantes para RAG

4. **Legibilidade**:
   - Nomes de variáveis em português misturados com inglês
   - Falta documentação e tipo hints

5. **Funcionalidade incompleta para RAG**:
   - Não extrai metadados (nome do arquivo, caminho, data)
   - Não processa formatos específicos (PDF, DOCX, etc.)
   - Não divide conteúdo em chunks para RAG

## Código revisado:

```python
import os
from pathlib import Path
from typing import List, Dict, Gen

---
## 7. Comparativo — qual técnica usar quando?

Vamos testar todas as técnicas no mesmo problema e comparar:

In [18]:
PROBLEMA = "Meu modelo de classificação tem 95% de acurácia no treino mas 60% no teste."

tecnicas = [
    (
        "Zero-shot",
        PROBLEMA,
        None
    ),
    (
        "Role Prompting",
        PROBLEMA,
        "Você é um especialista em diagnóstico de modelos de ML. Identifique o problema e dê 3 soluções práticas."
    ),
    (
        "Chain-of-Thought",
        f"""{PROBLEMA}
        
Raciocine passo a passo:
1. Qual é o nome técnico desse fenômeno?
2. Por que acontece?
3. Como diagnosticar a causa raiz?
4. Quais são as 3 melhores soluções?""",
        "Você é um especialista em ML."
    ),
]

for nome, prompt, system in tecnicas:
    print(f"\n{'='*55}")
    print(f"TÉCNICA: {nome}")
    print('='*55)
    r = chat(prompt, system=system, max_tokens=250)
    print(r)


TÉCNICA: Zero-shot
Isso é um **clássico caso de overfitting**. Seu modelo está memorizando os dados de treino em vez de aprender padrões generalizáveis. Vamos analisar causas e soluções:

## Possíveis causas:

1. **Modelo muito complexo** para a quantidade/qualidade dos dados
2. **Poucos dados de treino** em relação à complexidade do problema
3. **Vazamento de dados** (data leakage) - informações do teste vazando no treino
4. **Desbalanceamento de classes** que distorce a métrica
5. **Features irrelevantes ou ruidosas** sendo aprendidas

## Soluções para testar:

### 1. **Simplifique o modelo**
- Reduza a complexidade (profundidade de árvores, número de neurônios/camadas)
- Aumente a regularização (L1/L2, dropout)
- Use modelos mais simples como baseline (regressão logística)

### 2. **Melhore os dados**
- **Aumente o conjunto de treino** (data augmentation se for imagens/texto)
- **Validação cruz

TÉCNICA: Role Prompting
Identifiquei um caso clássico de **overfitting** (sobreajuste).

---
## Resumo

| Técnica | Quando usar | Custo extra |
|---|---|---|
| **Zero-shot** | Tarefas simples e diretas | Nenhum |
| **Few-shot** | Quando o formato de saída importa | Tokens dos exemplos |
| **Chain-of-Thought** | Problemas que exigem raciocínio | Tokens do raciocínio |
| **Role Prompting** | Quando tom e especialidade importam | Nenhum |
| **Output Formatting** | Quando a saída será processada por código | Nenhum |
| **Templates** | Prompts reutilizáveis no projeto | Nenhum |

### Regra prática
> Comece com **zero-shot**. Se a resposta não tiver o formato certo, adicione **output formatting**.  
> Se a qualidade não for boa, adicione **CoT** ou **few-shot**.

---